In [0]:

from pyspark.sql.functions import *

# Eventhub Configuration

event_hub_namespace = "rg-hospital-india.servicebus.windows.net"
event_hub_name = "rg-hospital-india"

event_hub_conn_str = dbutils.secrets.get(
    scope="hospitalval",
    key="eventhubconnection"
)

kafka_options = {
    "kafka.bootstrap.servers": f"{event_hub_namespace}:9093",
    "subscribe": event_hub_name,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{event_hub_conn_str}";',
    "startingOffsets": "latest",
    "failOnDataLoss": "false"
}

# Read from Eventhub

raw_df = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
)

# Cast data to JSON

json_df = raw_df.selectExpr("CAST(value AS STRING) AS raw_json")

# ADLS Configuration

bronze_path = "abfss://bronze@hospitalstorageaccoun.dfs.core.windows.net/patient_flow/"

# Write stream to Bronze

(
    json_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "abfss://bronze@hospitalstorageaccoun.dfs.core.windows.net/patient_flow/_checkpoints/"
    )
    .start(bronze_path)
)


In [0]:
manual_json = '{"patient_id": "767b9d0c-773f-492d-992f-36333513d22b", "gender": "Female", "age": 88, "department": "Emergency", "admission_time": "2025-08-19T09:59:39.147011", "discharge_time": "2025-08-21T11:59:39.147011", "bed_id": 335, "hospital_id": 5}'

manual_df = spark.createDataFrame([(manual_json,)], ["raw_json"])

manual_df.write.format("delta").mode("append").save(bronze_path)

In [0]:
display(spark.read.format("delta").load(bronze_path))

raw_json
"{""patient_id"": ""f0f3172a-3b33-410b-8889-8d80ddc79496"", ""gender"": ""Male"", ""age"": 22, ""department"": ""Oncology"", ""admission_time"": ""2026-11-08T17:10:06.318879"", ""discharge_time"": ""2026-11-11T08:10:06.318879"", ""bed_id"": 180, ""hospital_id"": 7}"
"{""patient_id"": ""7cb8ddc9-d23a-4395-ad4c-f5d37848227b"", ""gender"": ""Male"", ""age"": 46, ""department"": ""ICU"", ""admission_time"": ""2026-11-10T13:10:07.322284"", ""discharge_time"": ""2026-11-11T20:10:07.322284"", ""bed_id"": 340, ""hospital_id"": 5}"
"{""patient_id"": ""a0466e11-2e0b-461b-bbc6-7f1a95bd35e5"", ""gender"": ""Male"", ""age"": 126, ""department"": ""Oncology"", ""admission_time"": ""2026-11-10T03:10:08.326219"", ""discharge_time"": ""2026-11-12T01:10:08.326219"", ""bed_id"": 230, ""hospital_id"": 6}"
"{""patient_id"": ""700f9d3e-b907-4cd1-a8fa-896467be5fec"", ""gender"": ""Female"", ""age"": 76, ""department"": ""Surgery"", ""admission_time"": ""2026-11-08T21:10:09.329168"", ""discharge_time"": ""2026-11-11T00:10:09.329168"", ""bed_id"": 97, ""hospital_id"": 1}"
"{""patient_id"": ""c70819f3-3afe-47ce-bce1-6aa90e5cb2d7"", ""gender"": ""Female"", ""age"": 107, ""department"": ""Pediatrics"", ""admission_time"": ""2026-11-08T18:10:10.334658"", ""discharge_time"": ""2026-11-11T02:10:10.334658"", ""bed_id"": 104, ""hospital_id"": 6}"
"{""patient_id"": ""dc5e6dc9-2c17-402a-8aa1-c596229c1af3"", ""gender"": ""Male"", ""age"": 86, ""department"": ""Pediatrics"", ""admission_time"": ""2026-11-10T17:10:11.339316"", ""discharge_time"": ""2026-11-12T09:10:11.339316"", ""bed_id"": 410, ""hospital_id"": 7}"
"{""patient_id"": ""79bd0c76-366b-4024-937c-492685828d18"", ""gender"": ""Female"", ""age"": 106, ""department"": ""ICU"", ""admission_time"": ""2026-11-09T22:10:12.342683"", ""discharge_time"": ""2026-11-12T05:10:12.342683"", ""bed_id"": 450, ""hospital_id"": 7}"
"{""patient_id"": ""57b89dc3-0331-432a-a776-cd6f51f31e3a"", ""gender"": ""Male"", ""age"": 21, ""department"": ""ICU"", ""admission_time"": ""2026-11-09T06:10:13.348177"", ""discharge_time"": ""2026-11-09T22:10:13.348177"", ""bed_id"": 124, ""hospital_id"": 3}"
"{""patient_id"": ""d2956db9-72d5-47a2-81bb-a7e99b78dd0c"", ""gender"": ""Female"", ""age"": 97, ""department"": ""Maternity"", ""admission_time"": ""2026-11-09T07:10:14.354081"", ""discharge_time"": ""2026-11-10T10:10:14.354081"", ""bed_id"": 334, ""hospital_id"": 7}"
"{""patient_id"": ""6ff426f2-e63b-45ff-adbd-a17e3aad94b1"", ""gender"": ""Male"", ""age"": 33, ""department"": ""Surgery"", ""admission_time"": ""2026-11-09T01:10:15.356307"", ""discharge_time"": ""2026-11-10T09:10:15.356307"", ""bed_id"": 476, ""hospital_id"": 2}"
